In [15]:
import pandas as pd
import numpy as np
import sqlite3
import os

raw_path = r'C:\Users\86151\Desktop\就业档案0921\电商用户经营分析\data\raw'
clean_path = r'C:\Users\86151\Desktop\就业档案0921\电商用户经营分析\data\cleaned'
db_path = r'C:\Users\86151\Desktop\就业档案0921\电商用户经营分析\data\ecommerce.db'

users = pd.read_csv(os.path.join(raw_path, 'users.csv'))
products = pd.read_csv(os.path.join(raw_path, 'products.csv'))
orders = pd.read_csv(os.path.join(raw_path, 'orders.csv'))
daily_targets = pd.read_csv(os.path.join(raw_path, 'daily_targets.csv'))

print("原始数据:")
print(f"users: {users.shape}")
print(f"products: {products.shape}")
print(f"orders: {orders.shape}")
print(f"daily_targets: {daily_targets.shape}")

原始数据:
users: (1200, 6)
products: (60, 4)
orders: (7000, 12)
daily_targets: (365, 2)


In [16]:
users['city'] = users['city'].str.strip().str.replace('市', '', regex=False)
users['register_date'] = pd.to_datetime(users['register_date'])

print("清洗后城市:")
for c in sorted(users['city'].unique()):
    print(f"  [{c}]")

清洗后城市:
  [上海]
  [乌鲁木齐]
  [兰州]
  [北京]
  [南京]
  [南昌]
  [厦门]
  [哈尔滨]
  [大连]
  [天津]
  [广州]
  [成都]
  [昆明]
  [杭州]
  [武汉]
  [沈阳]
  [济南]
  [深圳]
  [福州]
  [苏州]
  [西安]
  [贵阳]
  [郑州]
  [重庆]
  [银川]
  [长春]
  [长沙]
  [青岛]


In [18]:
# 渠道：先去空格，再统一大写
orders['channel'] = orders['channel'].str.strip().str.upper()

# 用 replace 去掉所有空白字符（包括全角空格）
orders['payment_method'] = orders['payment_method'].astype(str).str.replace(r'\s+', '', regex=True)

# 日期转标准格式
orders['order_date'] = pd.to_datetime(orders['order_date'])

# 衍生字段
orders['year'] = orders['order_date'].dt.year
orders['month'] = orders['order_date'].dt.month
orders['quarter'] = orders['order_date'].dt.quarter
orders['weekday'] = orders['order_date'].dt.dayofweek

# 是否有效订单（已完成才算有效）
orders['is_valid'] = (orders['order_status'] == '已完成').astype(int)

# 检查渠道
print("清洗后渠道:")
for ch in sorted(orders['channel'].unique()):
    print(f"  [{ch}] 长度={len(ch)}")

# 检查支付方式（新增）
print("\n清洗后支付方式:")
for pm in sorted(orders['payment_method'].unique()):
    print(f"  [{pm}] 长度={len(pm)}")

清洗后渠道:
  [APP] 长度=3
  [PC] 长度=2
  [小程序] 长度=3
  [直播] 长度=2

清洗后支付方式:
  [微信] 长度=2
  [支付宝] 长度=3
  [银行卡] 长度=3


In [28]:
print("宽表里的支付方式:")
for pm in wide['payment_method'].unique():
    print(f"[{pm}] 长度={len(pm)}")

宽表里的支付方式:
[支付宝] 长度=3
[微信] 长度=2
[银行卡] 长度=3


In [29]:
wide = orders.merge(users, on='user_id', how='left') \
             .merge(products, on='product_id', how='left')

wide = wide.rename(columns={'unit_price_x': 'order_unit_price',
                            'unit_price_y': 'product_unit_price'})

print("宽表形状:", wide.shape)
print("宽表列名:", wide.columns.tolist())
print(wide.head())

宽表形状: (7000, 25)
宽表列名: ['order_id', 'user_id', 'order_date', 'product_id', 'quantity', 'channel', 'payment_method', 'order_status', 'order_unit_price', 'amount', 'net_amount', 'is_refund', 'year', 'month', 'quarter', 'weekday', 'is_valid', 'gender', 'age', 'region', 'city', 'register_date', 'category', 'product_name', 'product_unit_price']
   order_id  user_id order_date  product_id  quantity channel payment_method  \
0    500001    10960 2025-01-18       20037         2     APP            支付宝   
1    500002    10409 2025-06-18       20054         1      直播            支付宝   
2    500003    10194 2025-07-17       20049         1     小程序             微信   
3    500004    10235 2025-05-23       20058         1      直播             微信   
4    500005    10718 2025-02-06       20034         1      PC             微信   

  order_status  order_unit_price  amount  ...  weekday  is_valid  gender  age  \
0          已完成             24.16   48.32  ...        5         1       女   29   
1          已完成 

In [30]:
print("users关联失败:", wide['gender'].isnull().sum())
print("products关联失败:", wide['category'].isnull().sum())
print("\n金额分布:")
print(wide[['quantity', 'amount', 'net_amount']].describe())
print("\n订单状态分布:")
print(wide['order_status'].value_counts())

users关联失败: 0
products关联失败: 0

金额分布:
          quantity       amount   net_amount
count  7000.000000  7000.000000  7000.000000
mean      1.581143   154.307041   134.314484
std       0.727465   135.672363   136.179927
min       1.000000    15.000000     0.000000
25%       1.000000    66.180000    45.840000
50%       1.000000   111.050000    93.130000
75%       2.000000   201.940000   184.175000
max       3.000000  1000.440000  1000.440000

订单状态分布:
order_status
已完成    6101
已退款     536
已取消     363
Name: count, dtype: int64


In [31]:
wide.to_csv(os.path.join(clean_path, 'wide_table.csv'), index=False, encoding='utf-8-sig')
orders.to_csv(os.path.join(clean_path, 'orders_clean.csv'), index=False, encoding='utf-8-sig')
users.to_csv(os.path.join(clean_path, 'users_clean.csv'), index=False, encoding='utf-8-sig')

conn = sqlite3.connect(db_path)
wide.to_sql('wide_table', conn, if_exists='replace', index=False)
orders.to_sql('orders_clean', conn, if_exists='replace', index=False)
users.to_sql('users_clean', conn, if_exists='replace', index=False)
conn.close()

print("清洗完成，已保存")

清洗完成，已保存


In [33]:
# 重新读回来验证
test = pd.read_csv(os.path.join(clean_path, 'wide_table.csv'))
print("重新读取CSV后的支付方式:")
for pm in test['payment_method'].unique():
    print(f"[{pm}] 长度={len(pm)}")

重新读取CSV后的支付方式:
[支付宝] 长度=3
[微信] 长度=2
[银行卡] 长度=3
